In [89]:
import re
import os 
import pandas as pd
import numpy as np
import spacy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sentence_transformers import SentenceTransformer, InputExample, losses

import random

from torch.utils.data import DataLoader

### Utility Functions

In [4]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

def normalize_pair_df(df, s1_col, s2_col, label_col, pair_type, negation_type = None):
    out = pd.DataFrame({
        "sent1": df[s1_col].apply(clean_text),
        "sent2": df[s2_col].apply(clean_text),
        "label": df[label_col].astype(int),
        "pair_type": pair_type, 
    })

    if negation_type is not None:
        out["negation_type"] = df[negation_type]
    else:
        out["negation_type"] = None 
        
    out = out[(out["sent1"] != "") & (out["sent2"] != "")]
    out = out.drop_duplicates().reset_index(drop=True)
    return out

## Data Pre-processing

### Paraphrase Data - MRPC Dataset

In [5]:
rows = []
file_path = "./Data/MRPC/msr_paraphrase_train.txt"

with open(file_path, "r", encoding = "utf-8", errors="replace") as f:
    header = f.readline().rstrip("\n").split("\t") 

    for line_num, line in enumerate(f, start=2):
        parts = line.rstrip("\n").split("\t", maxsplit=4)
        if(len(parts) == 5):
            rows.append(parts)
        else:
            print(f"Warning: Line {line_num} has {len(parts)} columns, expected 5. Skipping this line.")

paraphrase_df = pd.DataFrame(rows, columns=header)

In [6]:
paraphrase_df.shape

(4076, 5)

In [7]:
paraphrase_df.head()

,﻿Quality,#1 ID,#2 ID,#1 String,#2 String
0,1,702876,702977,"Amrozi accused his brother, whom he called ""th...","Referring to him as only ""the witness"", Amrozi..."
1,0,2108705,2108831,Yucaipa owned Dominick's before selling the ch...,Yucaipa bought Dominick's in 1995 for $693 mil...
2,1,1330381,1330521,They had published an advertisement on the Int...,"On June 10, the ship's owners had published an..."
3,0,3344667,3344648,"Around 0335 GMT, Tab shares were up 19 cents, ...","Tab shares jumped 20 cents, or 4.6%, to set a ..."
4,1,1236820,1236712,"The stock rose $2.11, or about 11 percent, to ...",PG&E Corp. shares jumped $1.63 or 8 percent to...


In [8]:
paraphrase_df.columns = paraphrase_df.columns.str.replace("\ufeff", "", regex=False).str.strip()
paraphrase_df["Quality"] = paraphrase_df["Quality"].astype(int)

In [9]:
paraphrase_df.to_csv("./Data/paraphrase_train.csv", index=False)

In [10]:
paraphrase_df.shape

(4076, 5)

In [11]:
paraphrase_df["Quality"].value_counts()

Quality
1    2753
0    1323
Name: count, dtype: int64

In [12]:
para_df = paraphrase_df[paraphrase_df["Quality"] == 1].copy()
para_df = normalize_pair_df(
    para_df, 
    s1_col = "#1 String", 
    s2_col = "#2 String", 
    label_col = "Quality", 
    pair_type = "paraphrase"
)

### Negation Data - Self Generated

#### Explicit Negations

In [90]:
nlp = spacy.load("en_core_web_sm")

BE_VERBS = {"am", "is", "are", "was", "were"}
MODALS = {"can", "could", "may", "might", "must", "shall", "should", "will", "would"}
HAVE_AUX = {"has", "have", "had"}
DO_AUX = {"do", "does", "did"}

LEXICAL_NEG_WORDS = {"never", "no", "nothing", "nobody", "neither", "hardly", "barely"}

CONTRACTION_MAP = {
    "isn't": ["is", "not"],
    "aren't": ["are", "not"],
    "wasn't": ["was", "not"],
    "weren't": ["were", "not"],
    "hasn't": ["has", "not"],
    "haven't": ["have", "not"],
    "hadn't": ["had", "not"],
    "won't": ["will", "not"],
    "wouldn't": ["would", "not"],
    "can't": ["can", "not"],
    "couldn't": ["could", "not"],
    "shouldn't": ["should", "not"],
    "don't": ["do", "not"],
    "doesn't": ["does", "not"],
    "didn't": ["did", "not"],
}

def untokenize(tokens):
    text = " ".join(tokens)
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)
    text = re.sub(r'"\s+', '"', text)
    text = re.sub(r"\s+\"", '"', text)
    return text

def expand_contractions(text):
    tokens = text.split()
    expanded = []
    for tok in tokens:
        low = tok.lower()
        stripped = re.sub(r"[^\w']","", low)
        if stripped in CONTRACTION_MAP:
            expanded.extend(CONTRACTION_MAP[stripped])
        else:
            expanded.append(tok)
    return untokenize(expanded)

def has_lexical_negation(doc):
    return any(tok.lower_ in LEXICAL_NEG_WORDS for tok in doc)

def flip_polarity(sentence):
    sentence = expand_contractions(sentence)
    doc = nlp(sentence)
    tokens = [tok.text for tok in doc]

    # 0. lexical negation -> skip
    if has_lexical_negation(doc):
        return None

    # 1. explicit syntactic negation: remove / restore
    neg_tokens = [tok for tok in doc if tok.dep_ == "neg" and tok.lower_ == "not"]
    if neg_tokens:
        neg_tok = neg_tokens[0]
        head = neg_tok.head  # auxiliary or verb being negated

        # remove "not"
        new_tokens = tokens[:neg_tok.i] + tokens[neg_tok.i + 1:]

        # Case A: do-support reversal
        # e.g. did not go -> went, does not like -> likes
        if head.lower_ in DO_AUX:
            # find main verb to the right
            main_verb = None
            for tok in doc:
                if tok.head == head and tok.pos_ == "VERB":
                    main_verb = tok
                    break
            if main_verb is None:
                return None

            lemma = main_verb.lemma_

            if head.lower_ == "did":
                restored = main_verb._.inflect("VBD") if main_verb._.inflect("VBD") else lemma
            elif head.lower_ == "does":
                restored = main_verb._.inflect("VBZ") if main_verb._.inflect("VBZ") else lemma
            else:  # do
                restored = lemma

            # remove aux and replace main verb
            tmp = tokens[:]
            tmp[main_verb.i] = restored
            tmp = [w for j, w in enumerate(tmp) if j not in {head.i, neg_tok.i}]
            return untokenize(tmp)

        # Case B: can not / will not / is not / has not -> just remove not
        return untokenize(new_tokens)

    # 2. no explicit negation -> add negation

    # modal auxiliary
    for tok in doc:
        if tok.lower_ in MODALS and tok.dep_ in {"aux", "auxpass"}:
            new_tokens = tokens[:]
            if tok.lower_ == "can":
                new_tokens[tok.i] = "cannot"
                return untokenize(new_tokens)
            new_tokens.insert(tok.i + 1, "not")
            return untokenize(new_tokens)

    # be verb
    for tok in doc:
        if tok.lower_ in BE_VERBS and tok.pos_ == "AUX":
            new_tokens = tokens[:]
            new_tokens.insert(tok.i + 1, "not")
            return untokenize(new_tokens)

    # have auxiliary
    for tok in doc:
        if tok.lower_ in HAVE_AUX and tok.dep_ in {"aux", "auxpass"}:
            new_tokens = tokens[:]
            new_tokens.insert(tok.i + 1, "not")
            return untokenize(new_tokens)

    # main verb do-support
    root = None
    for tok in doc:
        if tok.dep_ == "ROOT" and tok.pos_ == "VERB":
            root = tok
            break

    if root is None:
        return None

    lemma = root.lemma_
    tag = root.tag_

    if tag == "VBD":
        aux = ["did", "not"]
    elif tag == "VBZ":
        aux = ["does", "not"]
    elif tag in {"VB", "VBP"}:
        aux = ["do", "not"]
    else:
        return None

    new_tokens = tokens[:]
    new_tokens[root.i] = lemma
    new_tokens = new_tokens[:root.i] + aux + new_tokens[root.i:]
    return untokenize(new_tokens)

In [91]:
ori_s1_new = []
neg_s1_new = []
neg_labels = []

for idx, row in para_df.iterrows():
    sent1 = row["sent1"]
    sent2 = row["sent2"]

    neg_sent1 = flip_polarity(sent1)
    if neg_sent1 is not None:
        ori_s1_new.append(sent1)
        neg_s1_new.append(neg_sent1)
        neg_labels.append(0)  # label for negative pair

In [92]:
neg_new_df = pd.DataFrame({
    "sent1": ori_s1_new,
    "sent2": neg_s1_new,
    "label": neg_labels,
    "pair_type": "explicit_negation",
})

In [94]:
neg_new_df.head(4)

,sent1,sent2,label,pair_type
0,"Amrozi accused his brother, whom he called ""th...","Amrozi did not accuse his brother, whom he cal...",0,explicit_negation
1,They had published an advertisement on the Int...,They had not published an advertisement on the...,0,explicit_negation
2,"The stock rose $2.11, or about 11 percent, to ...","The stock did not rise $ 2.11, or about 11 per...",0,explicit_negation
3,Revenue in the first quarter of the year dropp...,Revenue in the first quarter of the year did n...,0,explicit_negation


In [95]:
neg_new_df.shape

(2584, 4)

In [96]:
neg_new_df = normalize_pair_df(
    neg_new_df,
    s1_col="sent1",
    s2_col="sent2",
    label_col="label",
    pair_type="explicit_negation"
)

In [97]:
neg_new_df.head(5)

,sent1,sent2,label,pair_type,negation_type
0,"Amrozi accused his brother, whom he called ""th...","Amrozi did not accuse his brother, whom he cal...",0,explicit_negation,None
1,They had published an advertisement on the Int...,They had not published an advertisement on the...,0,explicit_negation,None
2,"The stock rose $2.11, or about 11 percent, to ...","The stock did not rise $ 2.11, or about 11 per...",0,explicit_negation,None
3,Revenue in the first quarter of the year dropp...,Revenue in the first quarter of the year did n...,0,explicit_negation,None
4,The DVD-CCA then appealed to the state Supreme...,The DVD - CCA then did not appeal to the state...,0,explicit_negation,None


#### Implicit Negations

In [20]:
para_only = paraphrase_df[paraphrase_df["Quality"] == 1].copy()
sample_implicit = para_only.sample(n=150, random_state=42).reset_index(drop=True)
implicit_df = pd.DataFrame({
    "id": range(1, len(sample_implicit) + 1), 
    "s1": sample_implicit["#1 String"].values, 
    "implicit_negation": "", 
    "label": 0, 
    "negation_type": "", 
    "notes": ""
})

implicit_df.to_csv("./Data/implicit_negation_sample_100.csv", index=False, encoding="utf-8-sig")

In [21]:
im_neg_df = pd.read_csv("./Data/Implicit_negation_df_final.csv", encoding="latin1")

In [22]:
im_neg_df.shape

(150, 7)

In [23]:
im_neg_df.head(2)

,id,s1,implicit_negation,label,negation_type,notes,generated
0,1,The company must either pay NTP the $53.7-mill...,The company can avoid paying NTP the $53.7-mil...,0,verb_reversal,NaN,True
1,2,They said no decisions had been made about how...,They said final decisions had already been mad...,0,state_reversal,NaN,True


In [24]:
im_neg_df.notes.value_counts()

notes
IGNORE      15
CONSIDER     5
Name: count, dtype: int64

In [25]:
im_neg_df = im_neg_df[im_neg_df["generated"] == True]

In [26]:
i_neg_df = normalize_pair_df(
    im_neg_df, 
    s1_col="s1", 
    s2_col="implicit_negation", 
    label_col="label", 
    pair_type="implicit_negation", 
    negation_type="negation_type"
)

In [27]:
i_neg_df.head(2)

,sent1,sent2,label,pair_type,negation_type
0,The company must either pay NTP the $53.7-mill...,The company can avoid paying NTP the $53.7-mil...,0,implicit_negation,verb_reversal
1,They said no decisions had been made about how...,They said final decisions had already been mad...,0,implicit_negation,state_reversal


### Combine all Data

In [100]:
full_df = pd.concat([
    para_df[["sent1", "sent2", "label", "pair_type", "negation_type"]], 
    neg_new_df[["sent1", "sent2", "label", "pair_type", "negation_type"]], 
    i_neg_df[["sent1", "sent2", "label", "pair_type", "negation_type"]], 
], ignore_index = True)

In [101]:
full_df["pair_type"].value_counts()

pair_type
paraphrase           2753
explicit_negation    2584
implicit_negation     135
Name: count, dtype: int64

In [102]:
full_df["negation_type"].value_counts()

negation_type
antonym_substitution    69
verb_reversal           24
event_reversal          23
state_reversal           6
adverb_weakening         6
modal_verb_reversal      4
quantifier_reversal      2
numerical_negation       1
Name: count, dtype: int64

In [103]:
full_df = full_df.drop_duplicates().reset_index(drop=True)

print(full_df.shape)
print(full_df["label"].value_counts())
print(full_df["pair_type"].value_counts())
full_df.head()

(5472, 5)
label
1    2753
0    2719
Name: count, dtype: int64
pair_type
paraphrase           2753
explicit_negation    2584
implicit_negation     135
Name: count, dtype: int64


,sent1,sent2,label,pair_type,negation_type
0,"Amrozi accused his brother, whom he called ""th...","Referring to him as only ""the witness"", Amrozi...",1,paraphrase,None
1,They had published an advertisement on the Int...,"On June 10, the ship's owners had published an...",1,paraphrase,None
2,"The stock rose $2.11, or about 11 percent, to ...",PG&E Corp. shares jumped $1.63 or 8 percent to...,1,paraphrase,None
3,Revenue in the first quarter of the year dropp...,With the scandal hanging over Stewart's compan...,1,paraphrase,None
4,The DVD-CCA then appealed to the state Supreme...,The DVD CCA appealed that decision to the U.S....,1,paraphrase,None


In [104]:
unique_s1 = full_df["sent1"].unique()

train_s1, test_s1 = train_test_split(
    unique_s1,
    test_size=0.2,
    random_state=42
)

train_df = full_df[full_df["sent1"].isin(train_s1)].reset_index(drop=True)
test_df = full_df[full_df["sent1"].isin(test_s1)].reset_index(drop=True)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

overlap = set(train_df["sent1"]).intersection(set(test_df["sent1"]))
print("Overlap sent1:", len(overlap))

Train size: 4329
Test size: 1143
Overlap sent1: 0


In [105]:
print("Train pair types:\n", train_df["pair_type"].value_counts())
print("Test pair types:\n", test_df["pair_type"].value_counts())

Train pair types:
 pair_type
paraphrase           2203
explicit_negation    2065
implicit_negation      61
Name: count, dtype: int64
Test pair types:
 pair_type
paraphrase           550
explicit_negation    519
implicit_negation     74
Name: count, dtype: int64


## Performance Evaluation

In [106]:
def evaluate_predictions(y_true, y_pred, name="model", sub=False):
    if sub:
        acc = accuracy_score(y_true, y_pred)
        print(f"{name} Accuracy: {acc:.4f}")
        return acc 
    
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f"{name} Accuracy: {acc:.4f}")
    print(f"{name} F1:       {f1:.4f}")
    return acc, f1

def pair_features(emb1, emb2):
    return np.hstack([emb1, emb2, np.abs(emb1 - emb2)])

def compute_tfidf_cosine(df):
    s1 = df["sent1"].tolist()
    s2 = df["sent2"].tolist()
    vectorizer = TfidfVectorizer()
    vectorizer.fit(s1 + s2)

    v1 = vectorizer.transform(s1)
    v2 = vectorizer.transform(s2)

    sims = np.array([
        cosine_similarity(v1[i], v2[i])[0][0]
        for i in range(len(df))
    ])
    return sims

### Random Baseline

In [107]:
np.random.seed(42)

random_pred = np.random.randint(0, 2, size=len(test_df))
random_acc, random_f1 = evaluate_predictions(test_df["label"], random_pred, name="Random baseline")

Random baseline Accuracy: 0.5118
Random baseline F1:       0.5027


### Cosine Similarity Baseline with TF-IDF

In [108]:
train_sims = compute_tfidf_cosine(train_df)
test_sims = compute_tfidf_cosine(test_df)

best_t = None 
best_f1 = 0.0

for t in np.linspace(0.1, 0.9, 17):
    pred = (train_sims >= t).astype(int)
    f1 = f1_score(train_df["label"], pred)

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best TF-IDF cosine threshold: ", best_t)

Best TF-IDF cosine threshold:  0.1


In [109]:
cos_pred = (test_sims >= best_t).astype(int)
evaluate_predictions(test_df["label"], cos_pred, name="TF-IDF Cosine baseline")

TF-IDF Cosine baseline Accuracy: 0.4812
TF-IDF Cosine baseline F1:       0.6497


(0.48118985126859143, 0.6497341996455995)

### SBERT Embeddings

In [110]:
model = SentenceTransformer("all-MiniLM-L6-v2")

train_emb1 = model.encode(train_df["sent1"].tolist(), convert_to_numpy=True)
train_emb2 = model.encode(train_df["sent2"].tolist(), convert_to_numpy=True)

test_emb1 = model.encode(test_df["sent1"].tolist(), convert_to_numpy=True)
test_emb2 = model.encode(test_df["sent2"].tolist(), convert_to_numpy=True)

X_train = pair_features(train_emb1, train_emb2)
X_test = pair_features(test_emb1, test_emb2)

y_train = train_df["label"].values
y_test = test_df["label"].values

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [111]:
clf = LogisticRegression(max_iter=1000)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

evaluate_predictions(y_test, y_pred, name = "SBERT + Logistic Regression")
print(classification_report(y_test, y_pred))

SBERT + Logistic Regression Accuracy: 0.9239
SBERT + Logistic Regression F1:       0.9236
              precision    recall  f1-score   support

           0       0.96      0.89      0.92       593
           1       0.89      0.96      0.92       550

    accuracy                           0.92      1143
   macro avg       0.92      0.93      0.92      1143
weighted avg       0.93      0.92      0.92      1143



### Store Results

In [112]:
analysis_df = test_df.copy()
analysis_df["random_pred"] = random_pred
analysis_df["tfidf_cosine"] = test_sims
analysis_df["tfidf_pred"] = cos_pred
analysis_df["sbert_pred"] = y_pred

In [113]:
sbert_cosine = np.array([
    cosine_similarity(test_emb1[i].reshape(1, -1), test_emb2[i].reshape(1, -1))[0][0]
    for i in range(len(test_df))
])

analysis_df["sbert_cosine"] = sbert_cosine

In [114]:
analysis_df.head(3)

,sent1,sent2,label,pair_type,negation_type,random_pred,tfidf_cosine,tfidf_pred,sbert_pred,sbert_cosine
0,Wal-Mart said it would check all of its millio...,It has also said it would review all of its do...,1,paraphrase,None,0,0.402685,1,1,0.613726
1,"The songs are on offer for 99 cents each, or $...",The company will offer songs for 99 cents and ...,1,paraphrase,None,1,0.601378,1,1,0.904707
2,Comcast Class A shares were up 8 cents at $30....,The stock rose 48 cents to $30 yesterday in Na...,1,paraphrase,None,0,0.534827,1,1,0.608552


### Analysis Results

In [115]:
for subset in ["paraphrase", "explicit_negation", "implicit_negation"]:
    sub = analysis_df[analysis_df["pair_type"] == subset]
    print(f"---------- {subset} ----------")
    print("Count: ", len(sub))

    evaluate_predictions(sub["label"], sub["random_pred"], name="Random", sub = True)
    evaluate_predictions(sub["label"], sub["tfidf_pred"], name="TF-IDF Cosine", sub = True)
    evaluate_predictions(sub["label"], sub["sbert_pred"], name="SBERT + LR", sub = True)

---------- paraphrase ----------
Count:  550
Random Accuracy: 0.5127
TF-IDF Cosine Accuracy: 1.0000
SBERT + LR Accuracy: 0.9564
---------- explicit_negation ----------
Count:  519
Random Accuracy: 0.5145
TF-IDF Cosine Accuracy: 0.0000
SBERT + LR Accuracy: 0.9750
---------- implicit_negation ----------
Count:  74
Random Accuracy: 0.4865
TF-IDF Cosine Accuracy: 0.0000
SBERT + LR Accuracy: 0.3243


In [116]:
analysis_df["len1"] = analysis_df["sent1"].apply(lambda x: len(x.split()))
analysis_df["len2"] = analysis_df["sent2"].apply(lambda x: len(x.split()))
analysis_df["avg_len"] = (analysis_df["len1"] + analysis_df["len2"]) / 2

In [117]:
analysis_df.head(3)

,sent1,sent2,label,pair_type,negation_type,random_pred,tfidf_cosine,tfidf_pred,sbert_pred,sbert_cosine,len1,len2,avg_len
0,Wal-Mart said it would check all of its millio...,It has also said it would review all of its do...,1,paraphrase,None,0,0.402685,1,1,0.613726,17,22,19.5
1,"The songs are on offer for 99 cents each, or $...",The company will offer songs for 99 cents and ...,1,paraphrase,None,1,0.601378,1,1,0.904707,14,12,13.0
2,Comcast Class A shares were up 8 cents at $30....,The stock rose 48 cents to $30 yesterday in Na...,1,paraphrase,None,0,0.534827,1,1,0.608552,18,13,15.5


In [118]:
wrong_df = analysis_df[analysis_df["label"] != analysis_df["sbert_pred"]].copy()

print("Number of SBERT+LR errors:", len(wrong_df))
wrong_df[["sent1", "sent2", "label", "pair_type", "sbert_cosine"]].head(5)

Number of SBERT+LR errors: 87


,sent1,sent2,label,pair_type,sbert_cosine
23,"Georgia cannot afford to not get funding,"" sai...","Georgia cannot afford to not get funding,"" sai...",1,paraphrase,0.989125
57,"CAPPS II will not use bank records, records in...","CAPPS II will not use bank records, credit rec...",1,paraphrase,0.945095
79,None of Deans opponents picked him as someone ...,None of Dean's opponents picked him as someone...,1,paraphrase,0.960328
87,The law does not regulate how much individuals...,The law does not regulate how much individuals...,1,paraphrase,0.935010
104,Bond voiced disappointment that neither Presid...,Bond also voiced his disappointment that neith...,1,paraphrase,0.940591


### Fine-tune

In [119]:
print(train_df.columns)
print(test_df.columns)

print(train_df["label"].value_counts())
print(train_df["pair_type"].value_counts())

Index(['sent1', 'sent2', 'label', 'pair_type', 'negation_type'], dtype='object')
Index(['sent1', 'sent2', 'label', 'pair_type', 'negation_type'], dtype='object')
label
1    2203
0    2126
Name: count, dtype: int64
pair_type
paraphrase           2203
explicit_negation    2065
implicit_negation      61
Name: count, dtype: int64


In [120]:
train_examples = []

for _, row in train_df.iterrows():
    train_examples.append(
        InputExample(
            texts=[row["sent1"], row["sent2"]],
            label=float(row["label"])
        )
    )

print(len(train_examples))
print(train_examples[0])

4329
<InputExample> label: 1.0, texts: Amrozi accused his brother, whom he called "the witness", of deliberately distorting his evidence.; Referring to him as only "the witness", Amrozi accused his brother of deliberately distorting his evidence.


In [121]:
ft_model = SentenceTransformer("all-MiniLM-L6-v2")

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.CosineSimilarityLoss(ft_model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [122]:
output_path = "./fine_tuned_sbert_for_negation"

ft_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=2,
    warmup_steps=30,
    output_path="./fine_tuned_sbert_negation"
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.046707


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

#### Over-sample Implicit Negations

In [159]:
implicit_train = train_df[train_df["pair_type"] == "implicit_negation"].copy()
non_implicit_train = train_df[train_df["pair_type"] != "implicit_negation"].copy()

oversample_factor = 5
implicit_upsampled = pd.concat([implicit_train] * oversample_factor, ignore_index=True)

train_df_os = pd.concat([non_implicit_train, implicit_upsampled], ignore_index=True)
train_df_os = train_df_os.sample(frac = 1, random_state = 42).reset_index()

print(len(train_df_os))
print(train_df_os["pair_type"].value_counts())

4573
pair_type
paraphrase           2203
explicit_negation    2065
implicit_negation     305
Name: count, dtype: int64


In [160]:
train_examples_os = []

for _, row in train_df_os.iterrows():
    train_examples_os.append(
        InputExample(
            texts=[row["sent1"], row["sent2"]],
            label=float(row["label"])
        )
    )

print(len(train_examples_os))
print(train_examples_os[0])

4573
<InputExample> label: 1.0, texts: "I think it's going to be a close vote, but I think the grant proposal is going to win," McConnell said.; "I think it's going to be a close vote, but I think the grant proposal's going to win," said Sen. Mitch McConnell, assistant majority leader.


In [161]:
ft_model_os = SentenceTransformer("all-MiniLM-L6-v2")

train_dataloader_os = DataLoader(train_examples_os, shuffle=True, batch_size=16)
train_loss_os = losses.CosineSimilarityLoss(ft_model_os)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [162]:
output_path_os = "./fine_tuned_sbert_for_negation_over_sampled"

ft_model_os.fit(
    train_objectives=[(train_dataloader_os, train_loss_os)],
    epochs=2,
    warmup_steps=30,
    output_path=output_path_os
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.060975


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### Fine-tune Result Analysis

In [127]:
ft_model = SentenceTransformer("./fine_tuned_sbert_negation")
print("Fine-tuned model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Fine-tuned model loaded.


In [128]:
train_emb1_ft = ft_model.encode(train_df["sent1"].tolist(), convert_to_numpy=True)
train_emb2_ft = ft_model.encode(train_df["sent2"].tolist(), convert_to_numpy=True)
test_emb1_ft = ft_model.encode(test_df["sent1"].tolist(), convert_to_numpy=True)
test_emb2_ft = ft_model.encode(test_df["sent2"].tolist(), convert_to_numpy=True)

X_train_ft = pair_features(train_emb1_ft, train_emb2_ft)
X_test_ft = pair_features(test_emb1_ft, test_emb2_ft)

In [129]:
clf_ft = LogisticRegression(max_iter=1000, random_state=42)
clf_ft.fit(X_train_ft, y_train)

pred_ft = clf_ft.predict(X_test_ft)

In [130]:
evaluate_predictions(y_test, pred_ft, name = "Fine tune SBERT + Logistic Regression")
print(classification_report(y_test, pred_ft))

Fine tune SBERT + Logistic Regression Accuracy: 0.9361
Fine tune SBERT + Logistic Regression F1:       0.9369
              precision    recall  f1-score   support

           0       0.99      0.89      0.94       593
           1       0.89      0.99      0.94       550

    accuracy                           0.94      1143
   macro avg       0.94      0.94      0.94      1143
weighted avg       0.94      0.94      0.94      1143



In [131]:
# analysis_ft_df = test_df.copy()
# analysis_ft_df["ft_sbert_pred"] = pred_ft
analysis_df["ft_sbert_pred"] = pred_ft

In [132]:
sbert_ft_cosine = np.array([
    cosine_similarity(test_emb1_ft[i].reshape(1, -1), test_emb2_ft[i].reshape(1, -1))[0][0]
    for i in range(len(test_df))
])

analysis_df["sbert_ft_cosine"] = sbert_ft_cosine

In [133]:
for subset in ["paraphrase", "explicit_negation", "implicit_negation"]:
    sub = analysis_df[analysis_df["pair_type"] == subset]
    print(f"---------- {subset} ----------")
    print("Count: ", len(sub))

    evaluate_predictions(sub["label"], sub["ft_sbert_pred"], name="FT_SBERT + LR", sub = True)

---------- paraphrase ----------
Count:  550
FT_SBERT + LR Accuracy: 0.9855
---------- explicit_negation ----------
Count:  519
FT_SBERT + LR Accuracy: 0.9923
---------- implicit_negation ----------
Count:  74
FT_SBERT + LR Accuracy: 0.1757


#### Fine-tune with over-sampled

In [163]:
ft_model_os = SentenceTransformer(output_path_os)
print("Fine-tuned os model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Fine-tuned os model loaded.


In [164]:
train_emb1_ft_os = ft_model_os.encode(train_df["sent1"].tolist(), convert_to_numpy=True)
train_emb2_ft_os = ft_model_os.encode(train_df["sent2"].tolist(), convert_to_numpy=True)
test_emb1_ft_os = ft_model_os.encode(test_df["sent1"].tolist(), convert_to_numpy=True)
test_emb2_ft_os = ft_model_os.encode(test_df["sent2"].tolist(), convert_to_numpy=True)

X_train_ft_os = pair_features(train_emb1_ft_os, train_emb2_ft_os)
X_test_ft_os = pair_features(test_emb1_ft_os, test_emb2_ft_os)

In [165]:
clf_ft_os = LogisticRegression(max_iter=2000, random_state=42)
clf_ft_os.fit(X_train_ft_os, y_train)

pred_ft_os = clf_ft_os.predict(X_test_ft_os)

In [166]:
evaluate_predictions(y_test, pred_ft_os, name = "Fine tune os SBERT + Logistic Regression")
print(classification_report(y_test, pred_ft_os))

Fine tune os SBERT + Logistic Regression Accuracy: 0.9396
Fine tune os SBERT + Logistic Regression F1:       0.9399
              precision    recall  f1-score   support

           0       0.98      0.90      0.94       593
           1       0.90      0.98      0.94       550

    accuracy                           0.94      1143
   macro avg       0.94      0.94      0.94      1143
weighted avg       0.94      0.94      0.94      1143



In [167]:
# analysis_ft_os_df = test_df.copy()
# analysis_ft_os_df["ft_os_sbert_pred"] = pred_ft_os
analysis_df["ft_os_sbert_pred"] = pred_ft_os 

In [168]:
sbert_ft_os_cosine = np.array([
    cosine_similarity(test_emb1_ft_os[i].reshape(1, -1), test_emb2_ft_os[i].reshape(1, -1))[0][0]
    for i in range(len(test_df))
])

analysis_df["sbert_ft_os_cosine"] = sbert_ft_os_cosine

In [169]:
for subset in ["paraphrase", "explicit_negation", "implicit_negation"]:
    sub = analysis_df[analysis_df["pair_type"] == subset]
    print(f"---------- {subset} ----------")
    print("Count: ", len(sub))

    evaluate_predictions(sub["label"], sub["ft_os_sbert_pred"], name="FT_OS_SBERT + LR", sub = True)

---------- paraphrase ----------
Count:  550
FT_OS_SBERT + LR Accuracy: 0.9818
---------- explicit_negation ----------
Count:  519
FT_OS_SBERT + LR Accuracy: 0.9846
---------- implicit_negation ----------
Count:  74
FT_OS_SBERT + LR Accuracy: 0.3108


In [170]:
analysis_df

,sent1,sent2,label,pair_type,negation_type,random_pred,tfidf_cosine,tfidf_pred,sbert_pred,sbert_cosine,len1,len2,avg_len,ft_sbert_pred,sbert_ft_cosine,ft_os_sbert_pred,sbert_ft_os_cosine
0,Wal-Mart said it would check all of its millio...,It has also said it would review all of its do...,1,paraphrase,None,0,0.402685,1,1,0.613726,17,22,19.5,1,0.965594,1,0.931966
1,"The songs are on offer for 99 cents each, or $...",The company will offer songs for 99 cents and ...,1,paraphrase,None,1,0.601378,1,1,0.904707,14,12,13.0,1,0.976005,1,0.970421
2,Comcast Class A shares were up 8 cents at $30....,The stock rose 48 cents to $30 yesterday in Na...,1,paraphrase,None,0,0.534827,1,1,0.608552,18,13,15.5,1,0.953538,1,0.907812
3,Also demonstrating box-office strength _ and g...,Also demonstrating box-office strength -- and ...,1,paraphrase,None,0,0.894395,1,1,0.885153,25,25,25.0,1,0.990576,1,0.992298
4,The top rate will go to 4.45 percent for all r...,"For residents with incomes above $500,000, the...",1,paraphrase,None,0,0.677149,1,1,0.807038,16,14,15.0,1,0.967417,1,0.902309
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1138,While Bolton apparently fell and was immobiliz...,While Bolton apparently rose and was immobiliz...,0,implicit_negation,antonym_substitution,1,0.945940,1,1,0.966193,19,19,19.0,1,0.994495,1,0.961815
1139,Ridge said no actual explosives or other harmf...,Ridge said no actual explosives or other harmf...,0,implicit_negation,verb_reversal,0,0.890895,1,0,0.953657,12,14,13.0,1,0.963135,1,0.780487
1140,Upscale department store chain Nordstrom Inc. ...,Upscale department store chain Nordstrom Inc. ...,0,implicit_negation,antonym_substitution,1,0.901979,1,1,0.886260,19,19,19.0,1,0.982946,1,0.863275
1141,"""The benefit is equal for everyone, both in tr...","""The benefit is unequal for everyone, both in ...",0,implicit_negation,antonym_substitution,0,0.905474,1,0,0.958084,18,18,18.0,1,0.984402,1,0.856772


In [171]:
implicit_df = analysis_df[analysis_df["pair_type"] == "implicit_negation"]
explicit_df = analysis_df[analysis_df["pair_type"] == "explicit_negation"]

print("Baseline error:", 1 - accuracy_score(implicit_df["label"], implicit_df["sbert_pred"]))
print("FT error:", 1 - accuracy_score(implicit_df["label"], implicit_df["ft_sbert_pred"]))
print("FT+OS error:", 1 - accuracy_score(implicit_df["label"], implicit_df["ft_os_sbert_pred"]))

Baseline error: 0.6756756756756757
FT error: 0.8243243243243243
FT+OS error: 0.6891891891891893


#### Cosine Similarity Analysis

In [172]:
wrong_implicit = implicit_df[implicit_df["label"] != implicit_df["sbert_pred"]]

print("Wrong implicit cosine mean (base):", wrong_implicit["sbert_cosine"].mean())
print("Correct implicit cosine mean (base):", 
            implicit_df[implicit_df["label"] == implicit_df["sbert_pred"]]["sbert_cosine"].mean())

Wrong implicit cosine mean (base): 0.9179542
Correct implicit cosine mean (base): 0.9105198


In [173]:
wrong_implicit_ft = implicit_df[implicit_df["label"] != implicit_df["ft_sbert_pred"]]

print("Wrong implicit cosine mean (FT):", wrong_implicit["sbert_ft_cosine"].mean())
print("Correct implicit cosine mean (FT):", 
        implicit_df[implicit_df["label"] == implicit_df["ft_sbert_pred"]]["sbert_ft_cosine"].mean())

Wrong implicit cosine mean (FT): 0.84940344
Correct implicit cosine mean (FT): 0.28318083


In [174]:
wrong_implicit_ft = implicit_df[implicit_df["label"] != implicit_df["ft_os_sbert_pred"]]

print("Wrong implicit cosine mean (FT + OS):", wrong_implicit["sbert_ft_os_cosine"].mean())
print("Correct implicit cosine mean (FT + OS):", 
        implicit_df[implicit_df["label"] == implicit_df["ft_os_sbert_pred"]]["sbert_ft_os_cosine"].mean())

Wrong implicit cosine mean (FT + OS): 0.6504102
Correct implicit cosine mean (FT + OS): 0.10519197


In [175]:
wrong_explicit = explicit_df[explicit_df["label"] != explicit_df["sbert_pred"]]

print("Wrong data cosine mean (base):", wrong_explicit["sbert_cosine"].mean())
print("Correct data cosine mean (base):", 
        explicit_df[explicit_df["label"] == explicit_df["sbert_pred"]]["sbert_cosine"].mean())

Wrong data cosine mean (base): 0.9421482
Correct data cosine mean (base): 0.93546146


In [176]:
wrong_explicit_ft = explicit_df[explicit_df["label"] != explicit_df["ft_sbert_pred"]]

print("Wrong data cosine mean (FT):", wrong_explicit_ft["sbert_ft_cosine"].mean())
print("Correct data cosine mean (FT):", 
        explicit_df[explicit_df["label"] == explicit_df["ft_sbert_pred"]]["sbert_ft_cosine"].mean())

Wrong data cosine mean (FT): 0.9592243
Correct data cosine mean (FT): -0.0054508764


In [177]:
wrong_explicit_ft_os = explicit_df[explicit_df["label"] != explicit_df["ft_os_sbert_pred"]]

print("Wrong data cosine mean (FT + OS):", wrong_explicit_ft_os["sbert_ft_os_cosine"].mean())
print("Correct data cosine mean (FT + OS):", 
        explicit_df[explicit_df["label"] == explicit_df["ft_os_sbert_pred"]]["sbert_ft_os_cosine"].mean())

Wrong data cosine mean (FT + OS): 0.80054295
Correct data cosine mean (FT + OS): 0.011566673


In [178]:
# Similarity difference between paraphrase, explicit negation, and implicit negation. 

sim_mean_by_type = analysis_df.groupby("pair_type")["sbert_cosine"].mean()
print(sim_mean_by_type)

sim_mean_by_type_ft = analysis_df.groupby("pair_type")["sbert_ft_cosine"].mean()
print(sim_mean_by_type_ft)

sim_mean_by_type_ft_os = analysis_df.groupby("pair_type")["sbert_ft_os_cosine"].mean()
print(sim_mean_by_type_ft_os)

pair_type
explicit_negation    0.935629
implicit_negation    0.915543
paraphrase           0.838816
Name: sbert_cosine, dtype: float32
pair_type
explicit_negation    0.001984
implicit_negation    0.825303
paraphrase           0.958491
Name: sbert_ft_cosine, dtype: float32
pair_type
explicit_negation    0.023728
implicit_negation    0.557571
paraphrase           0.937371
Name: sbert_ft_os_cosine, dtype: float32


#### Sentence Length Analysis

In [179]:
wrong = analysis_df[analysis_df["label"] != analysis_df["sbert_pred"]]
correct = analysis_df[analysis_df["label"] == analysis_df["sbert_pred"]]

print("Wrong cases avg length:", wrong["avg_len"].mean())
print("Correct cases avg length:", correct["avg_len"].mean())

Wrong cases avg length: 20.379310344827587
Correct cases avg length: 20.226325757575758


In [180]:
wrong_explicit = explicit_df[explicit_df["label"] != explicit_df["sbert_pred"]]
correct_explicit = explicit_df[explicit_df["label"] == explicit_df["sbert_pred"]]

print("Wrong explicit avg length:", wrong_explicit["avg_len"].mean())
print("Correct explicit avg length:", correct_explicit["avg_len"].mean())

Wrong explicit avg length: 19.384615384615383
Correct explicit avg length: 20.83399209486166


In [181]:
wrong_implicit = implicit_df[implicit_df["label"] != implicit_df["sbert_pred"]]
correct_implicit = implicit_df[implicit_df["label"] == implicit_df["sbert_pred"]]

print("Wrong implicit avg length:", wrong_implicit["avg_len"].mean())
print("Correct implicit avg length:", correct_implicit["avg_len"].mean())

Wrong implicit avg length: 20.07
Correct implicit avg length: 18.479166666666668


In [182]:
wrong_implicit_ft = implicit_df[implicit_df["label"] != implicit_df["ft_sbert_pred"]]
correct_implicit_ft = implicit_df[implicit_df["label"] == implicit_df["ft_sbert_pred"]]

print("Wrong implicit avg length (FT):", wrong_implicit_ft["avg_len"].mean())
print("Correct implicit avg length (FT):", correct_implicit_ft["avg_len"].mean())

Wrong implicit avg length (FT): 20.049180327868854
Correct implicit avg length (FT): 17.23076923076923


In [183]:
wrong_implicit_ft_os = implicit_df[implicit_df["label"] != implicit_df["ft_os_sbert_pred"]]
correct_implicit_ft_os = implicit_df[implicit_df["label"] == implicit_df["ft_os_sbert_pred"]]

print("Wrong implicit avg length (FT+OS):", wrong_implicit_ft_os["avg_len"].mean())
print("Correct implicit avg length (FT+OS):", correct_implicit_ft_os["avg_len"].mean())

Wrong implicit avg length (FT+OS): 20.529411764705884
Correct implicit avg length (FT+OS): 17.391304347826086


#### Negation Type Analysis

In [184]:
print(train_df["negation_type"].value_counts())
print(test_df["negation_type"].value_counts())

negation_type
antonym_substitution    34
event_reversal          13
verb_reversal            7
modal_verb_reversal      3
state_reversal           3
quantifier_reversal      1
Name: count, dtype: int64
negation_type
antonym_substitution    35
verb_reversal           17
event_reversal          10
adverb_weakening         6
state_reversal           3
quantifier_reversal      1
modal_verb_reversal      1
numerical_negation       1
Name: count, dtype: int64


In [185]:
for t in implicit_df["negation_type"].unique():
    sub = implicit_df[implicit_df["negation_type"] == t]
    acc = accuracy_score(sub["label"], sub["sbert_pred"])
    print(t, acc)

verb_reversal 0.5882352941176471
state_reversal 0.0
antonym_substitution 0.17142857142857143
quantifier_reversal 0.0
modal_verb_reversal 0.0
adverb_weakening 1.0
event_reversal 0.1
numerical_negation 1.0


#### Examples

In [186]:
wrong_df = analysis_df[analysis_df["label"] != analysis_df["sbert_pred"]]

print(test_df["pair_type"].value_counts())
print(wrong_df["pair_type"].value_counts())

pair_type
paraphrase           550
explicit_negation    519
implicit_negation     74
Name: count, dtype: int64
pair_type
implicit_negation    50
paraphrase           24
explicit_negation    13
Name: count, dtype: int64


In [187]:
sample_df = (
    wrong_df
    .groupby("pair_type", group_keys = False)
    .sample(n=5, replace = True, random_state = 42)
    .reset_index(drop = True)
)

sample_df

,sent1,sent2,label,pair_type,negation_type,random_pred,tfidf_cosine,tfidf_pred,sbert_pred,sbert_cosine,len1,len2,avg_len,ft_sbert_pred,sbert_ft_cosine,ft_os_sbert_pred,sbert_ft_os_cosine
0,It would not affect economic damages such as l...,It would affect economic damages such as lost ...,0,explicit_negation,None,0,0.992888,1,1,0.884161,13,12,12.5,0,-0.008574,0,-0.059973
1,"""Contrary to the court's decision, we firmly b...","""Contrary to the court 's decision, we firmly ...",0,explicit_negation,None,1,0.994604,1,1,0.911998,21,21,21.0,0,0.018229,0,0.022992
2,He said there were complex reasons for the inc...,He said there were complex reasons for the inc...,0,explicit_negation,None,0,0.995176,1,1,0.962555,23,24,23.5,0,-0.018231,0,0.002570
3,Canada won't commit more troops to Afghanistan...,Canada will commit more troops to Afghanistan ...,0,explicit_negation,None,1,0.957275,1,1,0.956120,20,22,21.0,0,0.060403,0,0.049698
4,"""I don't want to be the candidate for guys wit...","""I do want to be the candidate for guys with C...",0,explicit_negation,None,0,0.949550,1,1,0.972750,21,20,20.5,0,0.027888,0,-0.002759
5,President Bush and top administration official...,President Bush and top administration official...,0,implicit_negation,event_reversal,1,0.918099,1,1,0.907454,22,22,22.0,1,0.946809,1,0.296195
6,"In a televised interview on Wednesday, ECB Pre...","In a televised interview on Wednesday, ECB Pre...",0,implicit_negation,state_reversal,1,0.901996,1,1,0.898823,26,27,26.5,1,0.968902,1,0.840237
7,"""Today, we are trying to convey this problem t...","""Today, we are avoiding any mention of this pr...",0,implicit_negation,verb_reversal,0,0.769893,1,1,0.883178,19,20,19.5,1,0.766625,0,0.162464
8,"Strikingly, the poll saw little difference bet...","Strikingly, the poll saw a sharp divide betwee...",0,implicit_negation,antonym_substitution,1,0.803615,1,1,0.818348,16,17,16.5,1,0.981122,1,0.953415
9,The dollar rose 0.6 percent to 109.54 yen<JPY=...,The dollar fell 0.6 percent to 109.54 yen<JPY=...,0,implicit_negation,antonym_substitution,0,0.869227,1,1,0.871497,18,18,18.0,1,0.965670,1,0.602414


### Contrastive Learning Fine-tune

In [192]:
triplet_df = (
    train_df 
    .pivot_table(
        index = 'sent1', 
        columns = 'label', 
        values = 'sent2',
        aggfunc = lambda x: x.sample(1, random_state=42).iloc[0]
    )
    .reset_index()
)


In [193]:
triplet_df

label,sent1,0,1
0,"""30 years waiting for you,"" read one banner ho...",NaN,"""30 years waiting for you,"" read one banner ho..."
1,"""A lot of these are made-up stories,"" Schwarze...","""A lot of these are not made - up stories,""Sch...","""Well, first of all, a lot of these are made u..."
2,"""APEC leaders are painfully aware that securit...","""APEC leaders are not painfully aware that sec...","""APEC leaders are painfully aware that securit..."
3,"""Admiral Black has provided spiritual guidance...","""Admiral Black has not provided spiritual guid...","""Admiral Black has provided spiritual guidance..."
4,"""After careful analysis, WHO has concluded tha...","""After careful analysis, WHO has concluded tha...","""After careful analysis, WHO has concluded tha..."
...,...,...,...
2199,"Yesterday, shares closed up 29 cents, or 0.54 ...","Yesterday, shares did not close up 29 cents, o...",Amazon's shares yesterday closed at $54.32 on ...
2200,"Yesterday, two Japanese diplomats were killed ...","Yesterday, two Japanese diplomats were not kil...",Two Japanese diplomats were also killed in an ...
2201,You have to dig deep and come up with the good...,You have to dig deep and come up with the good...,You have to dig deep and come up with the good...
2202,"Zulifquar Ali, a worshiper slightly wounded by...","Zulifquar Ali, a worshiper slightly wounded by...","Witness Zulfiqar Ali, who was slightly wounded..."


In [194]:
triplet_df[0][0]

nan

In [227]:
triplet_examples = []

for sent1, group in train_df.groupby("sent1"):
    pos = group[group["label"] == 1]["sent2"].dropna().tolist()
    neg = group[group["label"] == 0]["sent2"].dropna().tolist()

    if len(pos) > 0 and len(neg) > 0:
        for p in pos:
            for n in neg:
                triplet_examples.append(InputExample(texts=[sent1, p, n]))

In [228]:
print(len(triplet_examples))

2125


In [229]:
model_contrastive = SentenceTransformer("all-MiniLM-L6-v2")
train_dataloader_contrastive = DataLoader(triplet_examples, shuffle=True, batch_size=16)
train_loss_contrastive = losses.TripletLoss(model_contrastive)

model_contrastive.fit(
    train_objectives=[(train_dataloader_contrastive, train_loss_contrastive)],
    epochs=2,
    warmup_steps=30,
    output_path="./fine_tuned_sbert_contrastive"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

#### Analyze the result of contrastive learning fine-tune

In [230]:
ft_contrastive_model = SentenceTransformer("./fine_tuned_sbert_contrastive")
train_emb1_con = ft_contrastive_model.encode(train_df["sent1"].tolist(), convert_to_numpy=True)
train_emb2_con = ft_contrastive_model.encode(train_df["sent2"].tolist(), convert_to_numpy=True)
test_emb1_con = ft_contrastive_model.encode(test_df["sent1"].tolist(), convert_to_numpy=True)
test_emb2_con = ft_contrastive_model.encode(test_df["sent2"].tolist(), convert_to_numpy=True)

X_train_con = pair_features(train_emb1_con, train_emb2_con)
X_test_con = pair_features(test_emb1_con, test_emb2_con)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [231]:
clf_con = LogisticRegression(max_iter=1000, random_state=42)
clf_con.fit(X_train_con, y_train)
pred_con = clf_con.predict(X_test_con)

In [232]:
evaluate_predictions(y_test, pred_con, name = "Fine tune SBERT Contrastive + Logistic Regression")
print(classification_report(y_test, pred_con))

Fine tune SBERT Contrastive + Logistic Regression Accuracy: 0.9274
Fine tune SBERT Contrastive + Logistic Regression F1:       0.9213
              precision    recall  f1-score   support

           0       0.90      0.97      0.93       593
           1       0.96      0.88      0.92       550

    accuracy                           0.93      1143
   macro avg       0.93      0.93      0.93      1143
weighted avg       0.93      0.93      0.93      1143



In [233]:
analysis_df["con_sbert_pred"] = pred_con

In [234]:
for subset in ["paraphrase", "explicit_negation", "implicit_negation"]:
    sub = analysis_df[analysis_df["pair_type"] == subset]
    print(f"---------- {subset} ----------")
    print("Count: ", len(sub))

    evaluate_predictions(sub["label"], sub["sbert_pred"], name="Base SBERT + LR", sub = True)
    evaluate_predictions(sub["label"], sub["ft_sbert_pred"], name="FT_SBERT + LR", sub = True)
    evaluate_predictions(sub["label"], sub["ft_os_sbert_pred"], name="FT_OS_SBERT + LR", sub = True)
    evaluate_predictions(sub["label"], sub["con_sbert_pred"], name="FT_SBERT Contrastive + LR", sub = True)

---------- paraphrase ----------
Count:  550
Base SBERT + LR Accuracy: 0.9564
FT_SBERT + LR Accuracy: 0.9855
FT_OS_SBERT + LR Accuracy: 0.9818
FT_SBERT Contrastive + LR Accuracy: 0.8836
---------- explicit_negation ----------
Count:  519
Base SBERT + LR Accuracy: 0.9750
FT_SBERT + LR Accuracy: 0.9923
FT_OS_SBERT + LR Accuracy: 0.9846
FT_SBERT Contrastive + LR Accuracy: 0.9807
---------- implicit_negation ----------
Count:  74
Base SBERT + LR Accuracy: 0.3243
FT_SBERT + LR Accuracy: 0.1757
FT_OS_SBERT + LR Accuracy: 0.3108
FT_SBERT Contrastive + LR Accuracy: 0.8784


In [224]:
print(len(set(triplet_df["sent1"]).intersection(set(test_df["sent1"]))))

0


In [235]:
analysis_df["sbert_contrastive_cosine"] = np.array([
    cosine_similarity(
        test_emb1_con[i].reshape(1, -1), 
        test_emb2_con[i].reshape(1, -1)
    )[0][0]
    for i in range(len(test_df))
])

In [236]:
# Similarity difference between paraphrase, explicit negation, and implicit negation. 

sim_mean_by_type = analysis_df.groupby("pair_type")["sbert_cosine"].mean()
print(sim_mean_by_type)

sim_mean_by_type_ft = analysis_df.groupby("pair_type")["sbert_ft_cosine"].mean()
print(sim_mean_by_type_ft)

sim_mean_by_type_ft_os = analysis_df.groupby("pair_type")["sbert_ft_os_cosine"].mean()
print(sim_mean_by_type_ft_os)

sim_mean_by_type_con = analysis_df.groupby("pair_type")["sbert_contrastive_cosine"].mean()
print(sim_mean_by_type_con)

pair_type
explicit_negation    0.935629
implicit_negation    0.915543
paraphrase           0.838816
Name: sbert_cosine, dtype: float32
pair_type
explicit_negation    0.001984
implicit_negation    0.825303
paraphrase           0.958491
Name: sbert_ft_cosine, dtype: float32
pair_type
explicit_negation    0.023728
implicit_negation    0.557571
paraphrase           0.937371
Name: sbert_ft_os_cosine, dtype: float32
pair_type
explicit_negation    0.993111
implicit_negation    0.981605
paraphrase           0.516849
Name: sbert_contrastive_cosine, dtype: float32


In [245]:
implicit_df = analysis_df[analysis_df["pair_type"] == "implicit_negation"]
wrong_implicit_con = implicit_df[implicit_df["label"] != implicit_df["con_sbert_pred"]]

print("Wrong implicit cosine mean (Contrastive):", wrong_implicit_con["sbert_contrastive_cosine"].mean())
print("Correct implicit cosine mean (Contrastive):", 
        implicit_df[implicit_df["label"] == implicit_df["con_sbert_pred"]]["sbert_contrastive_cosine"].mean())

Wrong implicit cosine mean (Contrastive): 0.9154949
Correct implicit cosine mean (Contrastive): 0.99075884


In [246]:
para_df = analysis_df[analysis_df["pair_type"] == "paraphrase"]
wrong_para_con = para_df[para_df["label"] != para_df["con_sbert_pred"]]

print("Wrong paraphrase cosine mean (Contrastive):", wrong_para_con["sbert_contrastive_cosine"].mean())
print("Correct paraphrase cosine mean (Contrastive):", 
        para_df[para_df["label"] == para_df["con_sbert_pred"]]["sbert_contrastive_cosine"].mean())

Wrong paraphrase cosine mean (Contrastive): 0.98348945
Correct paraphrase cosine mean (Contrastive): 0.45539856


In [247]:
explicit_df = analysis_df[analysis_df["pair_type"] == "explicit_negation"]
wrong_explicit_con = explicit_df[explicit_df["label"] != explicit_df["con_sbert_pred"]]

print("Wrong explicit cosine mean (Contrastive):", wrong_explicit_con["sbert_contrastive_cosine"].mean())
print("Correct explicit cosine mean (Contrastive):", 
        explicit_df[explicit_df["label"] == explicit_df["con_sbert_pred"]]["sbert_contrastive_cosine"].mean())

Wrong explicit cosine mean (Contrastive): 0.92947227
Correct explicit cosine mean (Contrastive): 0.99436164


In [248]:
wrong_df_con = analysis_df[analysis_df["label"] != analysis_df["con_sbert_pred"]]
print(wrong_df_con["pair_type"].value_counts())

pair_type
paraphrase           64
explicit_negation    10
implicit_negation     9
Name: count, dtype: int64
